# Notebook 02 · Modelado

El dataset tiene un desbalance de 19:1 (95% sin ictus / 5% con ictus). Accuracy no es una métrica válida. Usamos AUC-ROC como métrica principal y Recall como métrica secundaria para minimizar falsos negativos.

## Índice
1. Setup e importaciones  
2. Configuración de MLflow  
3. Carga y versiones del dataset  
4. Preprocesado con ColumnTransformer  
5. Pipeline de entrenamiento  
6. Función principal `run_experiment`  
7. Ejecución de experimentos  
8. Comparativa de resultados

## 01 · Setup e importaciones

Cargamos todas las dependencias necesarias. Hay dos puntos clave aquí:
* Usamos <code>imblearn.pipeline.Pipeline</code> en lugar de <code>sklearn.pipeline.Pipeline</code> — esto es fundamental porque la versión de imbalanced-learn sabe que SMOTE solo debe aplicarse durante el <code>fit()</code>, nunca durante el <code>predict()</code> ni en el test set.<br><br>
* <code>warnings.filterwarnings('ignore')</code> limpia la salida del notebook — en producción esto nunca se haría.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

# ML — sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    recall_score,
    f1_score,
    precision_score,
    confusion_matrix,
    RocCurveDisplay
)

# XGBoost
from xgboost import XGBClassifier

# ⚠️ IMPORTANTE: usar imblearn.pipeline, NO sklearn.pipeline -- imblearn/Pipeline que entiende SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# MLflow
import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings("ignore")

# ── Constantes globales ──
RANDOM_STATE = 42
TEST_SIZE    = 0.2

print("✓ Importaciones completadas")

✓ Importaciones completadas


## 02 · Configuración de MLflow

Usamos MLflow para hacer tracking de experimentos. Cada vez que se entrena un modelo, MLflow guarda automáticamente los parámetros, métricas y el modelo serializado. Esto nos permite:
* Reproducir cualquier experimento anterior con exactamente los mismos parámetros
* Comparar todos los modelos del equipo en una sola tabla
* Versionar los modelos y descargar el mejor para producción

Para ver la UI ejecutar en terminal (Solo con WSL): <code>mlflow ui --backend-store-uri file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns --port 5001</code> y abrir <code>http://localhost:5001</code>


In [20]:
# ── Configuración MLflow ──
# Guardamos los experimentos en local (carpeta mlruns/)
MLFLOW_URI = "file:///mnt/c/Users/under/Documents/F5/3_Projects/Stroker_project/mlruns"

# ACUERDO DE EQUIPO: todos usamos exactamente este nombre
EXPERIMENT_NAME = "stroke_project_v2"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"✓ MLflow configurado")

✓ MLflow configurado


## 03 · Carga y versiones del dataset

<code>get_dataset(version)</code> centraliza toda la lógica de limpieza en un solo lugar. Si el equipo decide cambiar cómo se trata <code>work_type='children'</code>, 
solo hay que modificar esta función y todos los experimentos se actualizan automáticamente.

Ofrecemos dos versiones del dataset:
* full: todos los pacientes (incluye menores de 17 años)
* adults: solo pacientes ≥ 17 años (decisión del EDA)


In [21]:
# Carga inicial — el raw nunca se modifica
df = pd.read_csv("../data/raw/stroke_dataset.csv")
print(f"✓ Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"  Stroke=1 : {df['stroke'].sum()} ({df['stroke'].mean()*100:.1f}%)")
print(f"  Stroke=0 : {(df['stroke']==0).sum()} ({(1-df['stroke'].mean())*100:.1f}%)")

✓ Dataset cargado: 4,981 filas × 11 columnas
  Stroke=1 : 248 (5.0%)
  Stroke=0 : 4733 (95.0%)


In [22]:
def get_dataset(version: str = "full") -> pd.DataFrame:
    """
    Devuelve el dataset con las limpiezas acordadas en el EDA.
    
    Parámetros
    ----------
    version : str
        'full'   → todos los pacientes
        'adults' → solo pacientes con age >= 17
    
    Retorna
    -------
    pd.DataFrame con las transformaciones aplicadas
    
    Decisiones del EDA aplicadas aquí
    ----------------------------------
    - work_type='children' → 'not_applied' (no tiene sentido laboral para menores)
    - smoking_status='Unknown' se mantiene como categoría implícita de faltante
    """
    assert version in ("full", "adults"), f"version debe ser 'full' o 'adults', recibido: {version}"
    
    df_copy = df.copy()
    
    # Decisión EDA: work_type='children' → 'not_applied'
    df_copy.loc[df_copy["work_type"] == "children", "work_type"] = "not_applied"
    df_copy['group'] = np.where(df_copy['age'] < 17, 'children', 'adults')
    
    # Decisión EDA: filtrar menores si se pide versión adultos
    if version == "adults":
        before = len(df_copy)
        df_copy = df_copy[df_copy["age"] >= 17].copy()
        print(f"  Filtro adultos: {before - len(df_copy)} filas eliminadas")
    
    print(f"✓ Dataset '{version}': {df_copy.shape[0]:,} filas")
    return df_copy

## 04 · Preprocesado con ColumnTransformer

<code>ColumnTransformer</code> aplica transformaciones distintas a cada grupo de columnas en paralelo:<br><br>

* Numéricas → StandardScaler: centra en media 0 y escala a desviación 1. Necesario para Logistic Regression (sensible a la escala). Random Forest y XGBoost no lo necesitan pero no les perjudica.
* Categóricas → OneHotEncoder: convierte categorías en columnas binarias. <code>handle_unknown='ignore'</code> evita errores si en producción llega una categoría no vista en entrenamiento.

In [23]:
def build_preprocessor(X_train: pd.DataFrame) -> ColumnTransformer:
    """
    Construye el preprocesador sobre X_train únicamente.
    
    El fit del ColumnTransformer aprenderá media, std y categorías
    solo del conjunto de entrenamiento — nunca del test.
    
    Parámetros
    ----------
    X_train : pd.DataFrame
        Conjunto de entrenamiento (sin la columna target)
    
    Retorna
    -------
    ColumnTransformer sin fitear (el Pipeline lo fiteará en .fit())
    """
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()
    num_cols = X_train.select_dtypes(exclude="object").columns.tolist()
    
    print(f"  Columnas numéricas  : {num_cols}")
    print(f"  Columnas categóricas: {cat_cols}")
    
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"),  cat_cols),
    ])
    
    return preprocessor

## 05 · Pipeline de entrenamiento

El Pipeline encadena pasos en orden. Cuando llamamos a <code>pipeline.fit(X_train, y_train)</code>, cada paso se ejecuta en secuencia:

> prep → [smote] → model

La magia del Pipeline de imblearn es que durante el <code>predict()</code>, SMOTE simplemente no se ejecuta — solo actúa en el <code>fit()</code>. Esto garantiza que el test nunca se modifique.


In [24]:
def build_pipeline(model, preprocessor, use_smote: bool = False) -> Pipeline:
    """
    Construye el Pipeline de entrenamiento.
    
    Orden de pasos:
        1. prep  : ColumnTransformer (escala + encoding)
        2. smote : SMOTE (solo si use_smote=True, solo en fit)
        3. model : clasificador
        
    Parámetros
    ----------
    model        : estimador sklearn compatible
    preprocessor : ColumnTransformer ya construido
    use_smote    : bool — si True, añade SMOTE al pipeline
    """
    steps = [("prep", preprocessor)]
    
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    
    steps.append(("model", model))
    
    return Pipeline(steps)

## 06 · Función principal run_experiment

Esta función orquesta todo el flujo de un experimento:

**Métricas que logueamos en MLflow:**

* auc — métrica principal, robusta al desbalance
* recall — minimizar falsos negativos (crítico en contexto médico)
* f1 — balance entre precision y recall
* precision — de los que predecimos positivo, cuántos lo son

**Tags vs Params en MLflow:**
* mlflow.log_param() → hiperparámetros del modelo (numéricos, comparables)
* mlflow.set_tag() → metadatos del experimento (autor, dataset, versión)

In [25]:
def run_experiment(model, model_name: str, dataset_version: str = "full", use_smote: bool = False, author: str = "Jonathan") -> dict:
    """
    Entrena un modelo, evalúa y registra todo en MLflow.
    
    Parámetros
    ----------
    model           : estimador sklearn (LogisticRegression, RandomForest, etc.)
    model_name      : str — nombre del run en MLflow
    dataset_version : str — 'full' o 'adults'
    use_smote       : bool — si True aplica SMOTE en entrenamiento
    author          : str — nombre del miembro del equipo
    
    Retorna
    -------
    dict con métricas del experimento (para comparativa al final)
    """
    print(f"\n{'='*60}")
    print(f"  {model_name} | dataset={dataset_version} | SMOTE={use_smote}")
    print(f"{'='*60}")
    
    # ── Datos ──
    df_exp = get_dataset(dataset_version)
    X = df_exp.drop("stroke", axis=1)
    y = df_exp["stroke"]
    
    # Split estratificado: garantiza misma proporción de clases en train y test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        stratify=y,          # ← fundamental con datos desbalanceados
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    
    print(f"  Train: {len(X_train):,} filas | Test: {len(X_test):,} filas")
    print(f"  Stroke en train: {y_train.sum()} ({y_train.mean()*100:.1f}%)")

    # ── CLONAR MODELO (CLAVE) ─────────────────────────
    model = clone(model)
    
    # ── Ajuste especial para XGBoost ──
    if isinstance(model, XGBClassifier) and not use_smote:
        neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
        model.set_params(scale_pos_weight = neg / pos)

    # ── 2. Pipeline ──
    preprocessor = build_preprocessor(X_train)
    pipeline = build_pipeline(model, preprocessor, use_smote)
    

    # ── 3. Entrenamiento + evaluación dentro del run de MLflow ────
    with mlflow.start_run(run_name=f"{model_name}_{dataset_version}_smote={use_smote}_V1"):
        
        # Entrenamiento
        pipeline.fit(X_train, y_train)

        # Predicciones
        y_pred = pipeline.predict(X_test)

        # PROBABILITIES SAFE HANDLING
        if hasattr(pipeline[-1], "predict_proba"):
            y_prob = pipeline.predict_proba(X_test)[:, 1]
        else:
            y_prob = pipeline.decision_function(X_test)
        
        # ── Métricas ──
        auc       = roc_auc_score(y_test, y_prob)
        recall    = recall_score(y_test, y_pred)
        f1        = f1_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        
        # ── Log métricas (acuerdo de equipo: estos nombres exactos) ──
        mlflow.log_metric("auc",       round(auc, 4))
        mlflow.log_metric("recall",    round(recall, 4))
        mlflow.log_metric("f1",        round(f1, 4))
        mlflow.log_metric("precision", round(precision, 4))
        
        # ── Log hiperparámetros del modelo ──
        # get_params() devuelve todos los parámetros automáticamente
        params = {k: str(v) for k, v in model.get_params().items()}
        mlflow.log_params(params)
        
        # ── Tags: metadatos del experimento (no son hiperparámetros) ──
        mlflow.set_tag("author", author)
        mlflow.set_tag("dataset_version", dataset_version)
        mlflow.set_tag("use_smote", str(use_smote))
        mlflow.set_tag("model_type", model_name)
        
        # ── Artifacts
        BASE_DIR = Path().resolve().parent   # sube de notebooks/ al root
        assets_dir = BASE_DIR / "assets/v1"

        # Definir path
        cm_path = os.path.join(assets_dir, f"confusion_matrix_{model_name}_{dataset_version}_smote={use_smote}.png") 

        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                         xticklabels=["No Stroke", "Stroke"],
                         yticklabels=["No Stroke", "Stroke"])        
        plt.savefig(cm_path)
        plt.close()
        mlflow.log_artifact(cm_path)

        # Curva ROC 
        roc_path = os.path.join(assets_dir, f"roc_curve_{model_name}_{dataset_version}_smote={use_smote}.png") 

        RocCurveDisplay.from_predictions(y_test, y_prob)
        plt.savefig(roc_path)
        plt.close()
        mlflow.log_artifact(roc_path)

        # ── Guardar el pipeline completo (preprocesado + modelo) ──
        mlflow.sklearn.log_model(pipeline, artifact_path=model_name)
        
        # ── Output en notebook ──
        print(f"\n  AUC-ROC   : {auc:.4f}  ← métrica principal")
        print(f"  Recall    : {recall:.4f}  ← minimizar falsos negativos")
        print(f"  F1-score  : {f1:.4f}")
        print(f"  Precision : {precision:.4f}")
        print(f"\n{classification_report(y_test, y_pred, target_names=['Sin ictus', 'Ictus'])}")
        
        print(f"  ✓ Run registrado en MLflow")
        
        return {
            "modelo":   model_name,
            "dataset":  dataset_version,
            "smote":    use_smote,
            "auc":      round(auc, 4),
            "recall":   round(recall, 4),
            "f1":       round(f1, 4),
            "precision":round(precision, 4),
        }

## 07 · Ejecución de experimentos

Aquí definimos los modelos y lanzamos los experimentos. Cada llamada a <code>run_experiment</code> genera un run independiente en MLflow con sus propias métricas, parámetros y el modelo serializado.<br><br>

Guardamos todos los resultados en una lista para generar la comparativa al final (Local).

In [26]:
# ── Definición de modelos ────────────────────────────────────────
# class_weight='balanced' ajusta los pesos inversamente proporcional a la frecuencia de cada clase.

lr_model = LogisticRegression(
    class_weight="balanced",  # compensa el desbalance 19:1
    max_iter=1000,             # más iteraciones para datasets con escalas distintas
    random_state=RANDOM_STATE
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    use_label_encoder=False
)

print("✓ Modelos definidos")
print(f"  Logistic Regression  : {lr_model.get_params()}")
print(f"  Random Forest        : {rf_model.get_params()}")
print(f"  XGBoost              : {xgb_model.get_params()}")

✓ Modelos definidos
  Logistic Regression  : {'C': 1.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 1000, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
  Random Forest        : {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
  XGBoost              : {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8, 'device': None, 'early_stoppin

In [ ]:
# ── Lanzamiento de experimentos ──
# Guardamos cada resultado para la comparativa final
AUTHOR = "Jonathan"

results = []

# Baseline: Logistic Regression
results.append(run_experiment(lr_model, "LogisticRegression", "full",   use_smote=False, author=AUTHOR))
results.append(run_experiment(lr_model, "LogisticRegression", "full",   use_smote=True,  author=AUTHOR))
results.append(run_experiment(lr_model, "LogisticRegression", "adults", use_smote=False, author=AUTHOR))
results.append(run_experiment(lr_model, "LogisticRegression", "adults", use_smote=True, author=AUTHOR))

# Random Forest
results.append(run_experiment(rf_model, "RandomForest",       "full",   use_smote=False, author=AUTHOR))
results.append(run_experiment(rf_model, "RandomForest",       "full",   use_smote=True,  author=AUTHOR))
results.append(run_experiment(rf_model, "RandomForest",       "adults", use_smote=False,  author=AUTHOR))
results.append(run_experiment(rf_model, "RandomForest",       "adults", use_smote=True,  author=AUTHOR))

# XGBoost
results.append(run_experiment(xgb_model, "XGBoost", "full", use_smote=False, author=AUTHOR))
results.append(run_experiment(xgb_model, "XGBoost", "full", use_smote=True, author=AUTHOR))
results.append(run_experiment(xgb_model, "XGBoost", "adults", use_smote=False, author=AUTHOR))
results.append(run_experiment(xgb_model, "XGBoost", "adults", use_smote=True, author=AUTHOR))


  LogisticRegression | dataset=full | SMOTE=False
✓ Dataset 'full': 4,981 filas
  Train: 3,984 filas | Test: 997 filas
  Stroke en train: 198 (5.0%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status', 'group']


2026/04/17 20:41:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 20:41:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  AUC-ROC   : 0.8410  ← métrica principal
  Recall    : 0.8400  ← minimizar falsos negativos
  F1-score  : 0.2545
  Precision : 0.1500

              precision    recall  f1-score   support

   Sin ictus       0.99      0.75      0.85       947
       Ictus       0.15      0.84      0.25        50

    accuracy                           0.75       997
   macro avg       0.57      0.79      0.55       997
weighted avg       0.95      0.75      0.82       997

  ✓ Run registrado en MLflow

  LogisticRegression | dataset=full | SMOTE=True
✓ Dataset 'full': 4,981 filas
  Train: 3,984 filas | Test: 997 filas
  Stroke en train: 198 (5.0%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status', 'group']


2026/04/17 20:42:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 20:42:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  AUC-ROC   : 0.8371  ← métrica principal
  Recall    : 0.7600  ← minimizar falsos negativos
  F1-score  : 0.2317
  Precision : 0.1367

              precision    recall  f1-score   support

   Sin ictus       0.98      0.75      0.85       947
       Ictus       0.14      0.76      0.23        50

    accuracy                           0.75       997
   macro avg       0.56      0.75      0.54       997
weighted avg       0.94      0.75      0.82       997

  ✓ Run registrado en MLflow

  RandomForest | dataset=full | SMOTE=False
✓ Dataset 'full': 4,981 filas
  Train: 3,984 filas | Test: 997 filas
  Stroke en train: 198 (5.0%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status', 'group']


...


  AUC-ROC   : 0.7967  ← métrica principal
  Recall    : 0.2000  ← minimizar falsos negativos
  F1-score  : 0.1493
  Precision : 0.1190

              precision    recall  f1-score   support

   Sin ictus       0.96      0.92      0.94       947
       Ictus       0.12      0.20      0.15        50

    accuracy                           0.89       997
   macro avg       0.54      0.56      0.54       997
weighted avg       0.91      0.89      0.90       997

  ✓ Run registrado en MLflow

  XGBoost | dataset=adults | SMOTE=False
  Filtro adultos: 770 filas eliminadas
✓ Dataset 'adults': 4,211 filas
  Train: 3,368 filas | Test: 843 filas
  Stroke en train: 197 (5.8%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status', 'group']


2026/04/17 20:43:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 20:43:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  AUC-ROC   : 0.7861  ← métrica principal
  Recall    : 0.5306  ← minimizar falsos negativos
  F1-score  : 0.2374
  Precision : 0.1529

              precision    recall  f1-score   support

   Sin ictus       0.97      0.82      0.89       794
       Ictus       0.15      0.53      0.24        49

    accuracy                           0.80       843
   macro avg       0.56      0.67      0.56       843
weighted avg       0.92      0.80      0.85       843

  ✓ Run registrado en MLflow

  XGBoost | dataset=adults | SMOTE=True
  Filtro adultos: 770 filas eliminadas
✓ Dataset 'adults': 4,211 filas
  Train: 3,368 filas | Test: 843 filas
  Stroke en train: 197 (5.8%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status', 'group']


2026/04/17 20:43:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/17 20:43:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



  AUC-ROC   : 0.8003  ← métrica principal
  Recall    : 0.2857  ← minimizar falsos negativos
  F1-score  : 0.2569
  Precision : 0.2333

              precision    recall  f1-score   support

   Sin ictus       0.96      0.94      0.95       794
       Ictus       0.23      0.29      0.26        49

    accuracy                           0.90       843
   macro avg       0.59      0.61      0.60       843
weighted avg       0.91      0.90      0.91       843

  ✓ Run registrado en MLflow


## 8 Comparativa



In [28]:
df_results = pd.DataFrame(results)
df_results.sort_values("auc", ascending=False)

,modelo,dataset,smote,auc,recall,f1,precision
6,XGBoost,full,False,0.8419,0.6200,0.2616,0.1658
0,LogisticRegression,full,False,0.8410,0.8400,0.2545,0.1500
1,LogisticRegression,full,True,0.8371,0.7600,0.2317,0.1367
4,LogisticRegression,adults,False,0.8332,0.8163,0.2500,0.1476
3,RandomForest,full,True,0.8128,0.1000,0.1075,0.1163
2,RandomForest,full,False,0.8051,0.0000,0.0000,0.0000
9,XGBoost,adults,True,0.8003,0.2857,0.2569,0.2333
7,XGBoost,full,True,0.7967,0.2000,0.1493,0.1190
8,XGBoost,adults,False,0.7861,0.5306,0.2374,0.1529
5,RandomForest,adults,True,0.7734,0.1020,0.1266,0.1667


In [35]:
print(df_results.columns)

Index(['modelo', 'dataset', 'smote', 'auc', 'recall', 'f1', 'precision'], dtype='object')


In [ ]:
 # SMOTE vs NO SMOTE
df_results.groupby("smote")[["auc", "recall", "f1"]].mean()

,auc,recall,f1
smote,,,
False,0.82146,0.56138,0.2007
True,0.80406,0.28954,0.1744


In [37]:
df_results.groupby("modelo")[["auc", "recall"]].mean()

,auc,recall
modelo,,
LogisticRegression,0.83710,0.805433
RandomForest,0.79710,0.067333
XGBoost,0.80625,0.409075


In [32]:
df_results.groupby("dataset")[["auc", "recall"]].mean()

,auc,recall
dataset,,
adults,0.798250,0.43365
full,0.822433,0.42000


## 9 Optuna

In [42]:
import optuna
from sklearn.base import clone

mlflow.end_run()

In [43]:
def run_optuna_experiment(model, model_name, trial, dataset_version="full", use_smote=False):

    # Clonar modelo para evitar contaminación
    model = clone(model)

    # ── Hiperparámetros según modelo ──
    if model_name == "LogisticRegression":
        model.set_params(
            C=trial.suggest_float("C", 0.01, 10, log=True)
        )

    elif model_name == "RandomForest":
        model.set_params(
            n_estimators=trial.suggest_int("n_estimators", 50, 300),
            max_depth=trial.suggest_int("max_depth", 3, 15)
        )

    elif model_name == "XGBoost":
        model.set_params(
            n_estimators=trial.suggest_int("n_estimators", 100, 400),
            max_depth=trial.suggest_int("max_depth", 3, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0)
        )

    # ── IMPORTANTE: nested run ──
    with mlflow.start_run(nested=True):

        result = run_experiment(
            model,
            model_name=model_name,
            dataset_version=dataset_version,
            use_smote=use_smote,
            author="Jonathan"
        )

    return result["auc"]

In [44]:
def make_objective(model, model_name, dataset_version="full", use_smote=False):

    def objective(trial):
        return run_optuna_experiment(
            model,
            model_name,
            trial,
            dataset_version,
            use_smote
        )

    return objective

In [45]:
study_lr = optuna.create_study(direction="maximize")

study_lr.optimize(
    make_objective(lr_model, "LogisticRegression", "full", use_smote=False),
    n_trials=20
)

print("Best LR params:", study_lr.best_params)

[I 2026-04-17 21:15:01,611] A new study created in memory with name: no-name-077a906d-1e54-4650-a0b4-26d21b42f7ec
[W 2026-04-17 21:15:02,454] Trial 0 failed with parameters: {'C': 0.6534635695792926} because of the following error: Exception('Run with UUID 8bcf302984844342acc520d13890422f is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True').
Traceback (most recent call last):
  File "/home/under/miniconda3/envs/py310/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_189986/4173143079.py", line 4, in objective
    return run_optuna_experiment(
  File "/tmp/ipykernel_189986/1516000192.py", line 30, in run_optuna_experiment
    result = run_experiment(
  File "/tmp/ipykernel_189986/1213674901.py", line 51, in run_experiment
    with mlflow.start_run(run_name=f"{model_name}_{dataset_version}_smote={use_smote}_V1"


  LogisticRegression | dataset=full | SMOTE=False
✓ Dataset 'full': 4,981 filas
  Train: 3,984 filas | Test: 997 filas
  Stroke en train: 198 (5.0%)
  Columnas numéricas  : ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
  Columnas categóricas: ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status', 'group']


Exception: Run with UUID 8bcf302984844342acc520d13890422f is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [ ]:
study_rf = optuna.create_study(direction="maximize")

study_rf.optimize(
    make_objective(rf_model, "RandomForest", "full", use_smote=False),
    n_trials=20
)

print("Best RF params:", study_rf.best_params)

In [ ]:
study_xgb = optuna.create_study(direction="maximize")

study_xgb.optimize(
    make_objective(xgb_model, "XGBoost", "full", use_smote=False),
    n_trials=30
)

print("Best XGB params:", study_xgb.best_params)

In [ ]:
study_xgb_smote = optuna.create_study(direction="maximize")

study_xgb_smote.optimize(
    make_objective(xgb_model, "XGBoost", "full", use_smote=True),
    n_trials=20
)

In [ ]:
best_xgb = xgb_model.set_params(**study_xgb.best_params)

run_experiment(
    best_xgb,
    "XGBoost_BEST",
    dataset_version="full",
    use_smote=False
)